# radon v1

In [5]:
import subprocess
import os

# 로컬 저장소와 브랜치를 확인하고, 해당 브랜치에서 작업할 수 있도록 설정
local_dir = "C:/Users/user\Documents\hackathon2025"  # 로컬 저장소 디렉토리
file_name = "test.py"  # 분석할 파일 이름

# 로컬 디렉토리로 이동
os.chdir(local_dir)

# 현재 브랜치가 LJM인지 확인 후 이동
result = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True)
current_branch = result.stdout.strip()

if current_branch == "LJM":
    print("✅ LJM 브랜치에 있습니다!")
else:
    print("❌ LJM 브랜치에 없습니다. LJM 브랜치로 전환합니다.")
    subprocess.run(["git", "checkout", "LJM"], capture_output=True, text=True)

# git pull로 최신화
subprocess.run(["git", "pull", "origin", "LJM"], capture_output=True, text=True)
print("✅ LJM 브랜치 최신화 완료!")

# radon cc를 사용해 파일 분석
def analyze_radon(file_path):
    """
    주어진 파일을 radon으로 분석하고 복잡도를 출력합니다.
    """
    try:
        cmd = ["radon", "cc", file_path, "-a", "-s"]
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", shell=True)
        print("📊 radon 코드 복잡도 분석 결과:")
        print(result.stdout)

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# 분석할 파일 경로 설정
file_path = os.path.join(local_dir, file_name)

# 파일 분석 실행
analyze_radon(file_path)


✅ LJM 브랜치에 있습니다!
✅ LJM 브랜치 최신화 완료!
📊 radon 코드 복잡도 분석 결과:
C:/Users/user\Documents\hackathon2025\test.py
    M 29:4 card.use_item - C (16)
    C 6:0 card - A (5)
    M 88:4 card.stuned - A (5)
    M 16:4 card.attack - A (2)
    M 77:4 card.check_item - A (2)
    M 7:4 card.__init__ - A (1)
    M 113:4 card.stun_down - A (1)
    M 117:4 card.stun_check - A (1)

8 blocks (classes, functions, methods) analyzed.
Average complexity: A (4.125)



# radon v2(점수 시각화 가독성)

In [5]:
import subprocess
import os
import re

# 📁 설정
local_dir = r"C:\Users\user\Documents\hackathon2025"  # 로컬 Git 저장소 경로
file_name = "test.py"  # 분석할 파일명
file_path = os.path.join(local_dir, file_name)  # 전체 경로

# 📍 디렉토리 이동
os.chdir(local_dir)

# 🔄 브랜치 확인 및 전환
branch_result = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               capture_output=True, text=True)
current_branch = branch_result.stdout.strip()

if current_branch != "LJM":
    print(f"❌ 현재 브랜치: {current_branch}")
    print("🔁 LJM 브랜치로 전환합니다.")
    subprocess.run(["git", "checkout", "LJM"], capture_output=True, text=True)
else:
    print("✅ 현재 브랜치: LJM")

# 🔄 최신화
subprocess.run(["git", "pull", "origin", "LJM"], capture_output=True, text=True)
print("✅ LJM 브랜치 최신화 완료!")

# 🧮 등급 → 항목별 점수 변환 함수
def grade_to_scores(grade):
    scores = {
        'A': [100, 100, 100, 100, 100],
        'B': [90, 90, 90, 90, 90],
        'C': [70, 70, 70, 70, 70],
        'D': [50, 40, 40, 40, 40],
        'E': [30, 20, 20, 20, 20],
        'F': [10, 10, 10, 10, 10],
    }
    return scores.get(grade.upper(), [0, 0, 0, 0, 0])

# 📊 점수 분석 함수
def analyze_scores(radon_output):
    grades = re.findall(r'\b[A-F]\b', radon_output)
    if not grades:
        print("❗ 등급 정보를 찾을 수 없습니다.")
        return

    total_scores = [0] * 5
    for grade in grades:
        grade_scores = grade_to_scores(grade)
        total_scores = [sum(x) for x in zip(total_scores, grade_scores)]

    avg_scores = [round(s / len(grades), 2) for s in total_scores]
    labels = ["구조 복잡도", "테스트 용이성", "유지 보수성", "가독성", "코드 품질"]

    print("\n📈 항목별 평균 점수:")
    for label, score in zip(labels, avg_scores):
        print(f"• {label}: {score}/100")

# 📦 radon 분석 실행
def analyze_radon(file_path):
    try:
        cmd = ["radon", "cc", file_path, "-a", "-s"]
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", shell=True)
        output = result.stdout

#         print("📊 radon 코드 복잡도 분석 결과:")
#         print(output)

#         with open("radon_result.txt", "w", encoding="utf-8") as f:
#             f.write(output)
#         print("📁 결과가 'radon_result.txt' 파일로 저장되었습니다!")

        analyze_scores(output)

    except Exception as e:
        print(f"❌ 오류 발생: {e}")

# 🚀 실행
print(f"\n📂 분석 대상 파일: {file_path}")
analyze_radon(file_path)


✅ 현재 브랜치: LJM
✅ LJM 브랜치 최신화 완료!

📂 분석 대상 파일: C:\Users\user\Documents\hackathon2025\test.py
📁 결과가 'radon_result.txt' 파일로 저장되었습니다!

📈 항목별 평균 점수:
• 구조 복잡도: 91.82/100
• 테스트 용이성: 91.82/100
• 유지 보수성: 91.82/100
• 가독성: 91.82/100
• 코드 품질: 91.82/100


# radon v3(점수를 변수로 저장)

In [1]:
import subprocess
import os
import re

# 📁 설정
local_dir = r"C:\Users\user\Documents\hackathon2025"
file_name = input("파일 이름 입력: ")
file_path = os.path.join(local_dir, file_name)

# 📍 디렉토리 이동
os.chdir(local_dir)

# 🔄 브랜치 확인 및 전환
branch_result = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               capture_output=True, text=True)
current_branch = branch_result.stdout.strip()

if current_branch != "LJM":
    print(f"❌ 현재 브랜치: {current_branch}")
    print("🔁 LJM 브랜치로 전환합니다.")
    subprocess.run(["git", "checkout", "LJM"], capture_output=True, text=True)
else:
    print("✅ 현재 브랜치: LJM")

# 🔄 최신화
subprocess.run(["git", "pull", "origin", "LJM"], capture_output=True, text=True)
print("✅ LJM 브랜치 최신화 완료!")

# 🧮 등급 → 점수 변환 함수
def grade_to_scores(grade):
    scores = {
        'A': [100, 100, 100, 100, 100],
        'B': [90, 90, 90, 90, 90],
        'C': [70, 70, 70, 70, 70],
        'D': [50, 40, 40, 40, 40],
        'E': [30, 20, 20, 20, 20],
        'F': [10, 10, 10, 10, 10],
    }
    return scores.get(grade.upper(), [0, 0, 0, 0, 0])

# 📊 점수 분석 함수
def analyze_scores(radon_output):
    grades = re.findall(r'\b[A-F]\b', radon_output)
    if not grades:
        print("❗ 등급 정보를 찾을 수 없습니다.")
        return [0, 0, 0, 0, 0]

    total_scores = [0] * 5
    for grade in grades:
        grade_scores = grade_to_scores(grade)
        total_scores = [sum(x) for x in zip(total_scores, grade_scores)]

    avg_scores = [round(s / len(grades), 2) for s in total_scores]
    return avg_scores

# 📦 radon 분석 실행
def analyze_radon(file_path):
    try:
        cmd = ["radon", "cc", file_path, "-a", "-s"]
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", shell=True)
        output = result.stdout

        print("📊 radon 코드 복잡도 분석 결과:")
#         print(output)

        # 점수 분석
        scores = analyze_scores(output)

        # 점수 변수 저장
        structure_score = scores[0]
        testability_score = scores[1]
        maintainability_score = scores[2]
        readability_score = scores[3]
        quality_score = scores[4]

        # 출력
        print("\n📈 항목별 평균 점수:")
        print(f"• 구조 복잡도: {structure_score}/10")
        print(f"• 테스트 용이성: {testability_score}/10")
        print(f"• 유지 보수성: {maintainability_score}/10")
        print(f"• 가독성: {readability_score}/10")
        print(f"• 코드 품질: {quality_score}/10")

        # 반환할 수도 있음
        return {
            "structure": structure_score,
            "testability": testability_score,
            "maintainability": maintainability_score,
            "readability": readability_score,
            "quality": quality_score
        }

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        return None

# 🚀 실행
print(f"\n📂 분석 대상 파일: {file_path}")
score_dict = analyze_radon(file_path)


파일 이름 입력:  s


FileNotFoundError: [WinError 2] 지정된 파일을 찾을 수 없습니다: 'C:\\Users\\user\\Documents\\hackathon2025'

# radon v4(점수 정확성 개선)

In [7]:
import subprocess
import os
import re

# 📁 설정
local_dir = r"C:/Users/USER/PycharmProjects/hackathon2025"
file_name = input("파일 이름 입력: ")
file_path = os.path.join(local_dir, file_name)

# 📍 디렉토리 이동
os.chdir(local_dir)

# 🔄 브랜치 확인 및 전환
branch_result = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               capture_output=True, text=True)
current_branch = branch_result.stdout.strip()

if current_branch != "LJM":
    print(f"❌ 현재 브랜치: {current_branch}")
    print("🔁 LJM 브랜치로 전환합니다.")
    subprocess.run(["git", "checkout", "LJM"], capture_output=True, text=True)
else:
    print("✅ 현재 브랜치: LJM")

# 🔄 최신화
subprocess.run(["git", "pull", "origin", "LJM"], capture_output=True, text=True)
print("✅ LJM 브랜치 최신화 완료!")

# 🧮 등급 → 점수 변환 함수
def grade_to_scores(grade):
    scores = {
        'A': [100, 95, 90, 95, 100],
        'B': [85, 80, 75, 85, 90],
        'C': [70, 65, 60, 70, 75],
        'D': [50, 40, 40, 50, 55],
        'E': [30, 25, 20, 30, 35],
        'F': [10, 10, 10, 10, 10],
    }
    return scores.get(grade.upper(), [0, 0, 0, 0, 0])

# 📊 점수 분석 함수
def analyze_scores(radon_output):
    grades = re.findall(r'\b[A-F]\b', radon_output)
    if not grades:
        print("❗ 등급 정보를 찾을 수 없습니다.")
        return [0, 0, 0, 0, 0]

    total_scores = [0] * 5
    for grade in grades:
        grade_scores = grade_to_scores(grade)
        total_scores = [sum(x) for x in zip(total_scores, grade_scores)]

    avg_scores = [round(s / len(grades), 2) for s in total_scores]
    return avg_scores

# 📦 radon 분석 실행
def analyze_radon(file_path):
    try:
        cmd = ["radon", "cc", file_path, "-a", "-s"]
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", shell=True)
        output = result.stdout

        print("📊 radon 코드 복잡도 분석 결과:")
        # print(output)

        # 점수 분석
        scores = analyze_scores(output)

        # 점수 변수 저장
        structure_score = scores[0]
        testability_score = scores[1]
        maintainability_score = scores[2]
        readability_score = scores[3]
        quality_score = scores[4]

        # 출력
        print("\n📈 항목별 평균 점수:")
        print(f"• 구조 복잡도: {structure_score}/100")
        print(f"• 테스트 용이성: {testability_score}/100")
        print(f"• 유지 보수성: {maintainability_score}/100")
        print(f"• 가독성: {readability_score}/100")
        print(f"• 코드 품질: {quality_score}/100")

        # 반환할 수도 있음
        return {
            "structure": structure_score,
            "testability": testability_score,
            "maintainability": maintainability_score,
            "readability": readability_score,
            "quality": quality_score
        }

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        return None

# 🚀 실행
print(f"\n📂 분석 대상 파일: {file_path}")
score_dict = analyze_radon(file_path)


파일 이름 입력:  test.py


✅ 현재 브랜치: LJM
✅ LJM 브랜치 최신화 완료!

📂 분석 대상 파일: C:/Users/USER/PycharmProjects/hackathon2025\test.py
📊 radon 코드 복잡도 분석 결과:

📈 항목별 평균 점수:
• 구조 복잡도: 91.82/100
• 테스트 용이성: 86.82/100
• 유지 보수성: 81.82/100
• 가독성: 88.18/100
• 코드 품질: 93.18/100


# radon v5

In [10]:
import subprocess
import os
import re

# 📁 설정
local_dir = r"C:/Users/USER/PycharmProjects/hackathon2025"
file_name = input("파일 이름 입력: ")
file_path = os.path.join(local_dir, file_name)

# 📍 디렉토리 이동
os.chdir(local_dir)

# 🔄 브랜치 확인 및 전환
branch_result = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                               capture_output=True, text=True)
current_branch = branch_result.stdout.strip()

if current_branch != "LJM":
    print(f"❌ 현재 브랜치: {current_branch}")
    print("🔁 LJM 브랜치로 전환합니다.")
    subprocess.run(["git", "checkout", "LJM"], capture_output=True, text=True)
else:
    print("✅ 현재 브랜치: LJM")

# 🔄 최신화
subprocess.run(["git", "pull", "origin", "LJM"], capture_output=True, text=True)
print("✅ LJM 브랜치 최신화 완료!")

# 🧮 등급 → 점수 변환 함수
def grade_to_scores(grade):
    scores = {
        'A': [100, 95, 90, 95, 100],
        'B': [85, 80, 75, 85, 90],
        'C': [70, 65, 60, 70, 75],
        'D': [50, 40, 40, 50, 55],
        'E': [30, 25, 20, 30, 35],
        'F': [10, 10, 10, 10, 10],
    }
    return scores.get(grade.upper(), [0, 0, 0, 0, 0])
# ✅ 점수 → 등급 문자 역변환 함수
def score_to_grade(score):
    if score >= 90:
        return "A"
    elif score >= 70:
        return "B"
    elif score >= 50:
        return "C"
    elif score >= 30:
        return "D"
    elif score >= 10:
        return "E"
    else:
        return "F"

# 📊 점수 분석 함수 (정수화 및 등급 매핑 포함)
def analyze_scores(radon_output):
    grades = re.findall(r'\b[A-F]\b', radon_output)
    if not grades:
        print("❗ 등급 정보를 찾을 수 없습니다.")
        return [0, 0, 0, 0, 0]

    total_scores = [0] * 5
    for grade in grades:
        grade_scores = grade_to_scores(grade)
        total_scores = [sum(x) for x in zip(total_scores, grade_scores)]

    avg_scores = [round(s / len(grades)) for s in total_scores]  # 정수 점수
    return avg_scores

# 📦 radon 분석 실행
def analyze_radon(file_path):
    try:
        cmd = ["radon", "cc", file_path, "-a", "-s"]
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", shell=True)
        output = result.stdout

        print("📊 radon 코드 복잡도 분석 결과:")
        print(output)

        # 점수 분석
        scores = analyze_scores(output)

        # 점수 변수 저장
        structure_score = scores[0]
        testability_score = scores[1]
        maintainability_score = scores[2]
        readability_score = scores[3]
        quality_score = scores[4]

        # 등급 매핑
        structure_grade = score_to_grade(structure_score)
        testability_grade = score_to_grade(testability_score)
        maintainability_grade = score_to_grade(maintainability_score)
        readability_grade = score_to_grade(readability_score)
        quality_grade = score_to_grade(quality_score)

        # 출력
        print("\n📈 항목별 평가 결과:")
        print(f"• 구조 복잡도: {structure_score}점 ({structure_grade} 등급)")
        print(f"• 테스트 용이성: {testability_score}점 ({testability_grade} 등급)")
        print(f"• 유지 보수성: {maintainability_score}점 ({maintainability_grade} 등급)")
        print(f"• 가독성: {readability_score}점 ({readability_grade} 등급)")
        print(f"• 코드 품질: {quality_score}점 ({quality_grade} 등급)")

        return {
            "structure": structure_score,
            "testability": testability_score,
            "maintainability": maintainability_score,
            "readability": readability_score,
            "quality": quality_score
        }

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        return None


# 🚀 실행
print(f"\n📂 분석 대상 파일: {file_path}")
score_dict = analyze_radon(file_path)





# 구조 복잡도: radon의 cyclomatic complexity 그대로.

# 테스트 용이성: 복잡도 낮을수록 유닛 테스트 작성이 쉽다는 가정.

# 유지 보수성: 함수의 복잡도와 크기(라인 수)가 작을수록 유리.

# 가독성: 복잡도가 낮으면 코드 흐름 파악이 쉬워서 높게 평가.

# 코드 품질: 위 요소들을 종합한 평균 점수로 판단.

파일 이름 입력:  test2.py


✅ 현재 브랜치: LJM
✅ LJM 브랜치 최신화 완료!

📂 분석 대상 파일: C:/Users/USER/PycharmProjects/hackathon2025\test2.py
📊 radon 코드 복잡도 분석 결과:
C:/Users/USER/PycharmProjects/hackathon2025\test2.py
    F 17:0 check_win - C (11)
    F 35:0 game_loop - B (10)
    F 4:0 print_board - A (3)
    F 11:0 is_valid_move - A (3)
    F 32:0 is_draw - A (3)
    F 14:0 place_mark - A (2)

6 blocks (classes, functions, methods) analyzed.
Average complexity: B (5.333333333333333)


📈 항목별 평가 결과:
• 구조 복잡도: 55점 (C 등급)
• 테스트 용이성: 52점 (C 등급)
• 유지 보수성: 49점 (D 등급)
• 가독성: 54점 (C 등급)
• 코드 품질: 56점 (C 등급)
